In [ ]:
"""
Micromouse NSGA Optimization - Full Analysis Pipeline
======================================================
Tracks 1-3 with increasing curve sharpness (skew).

Sections
--------
0.  Imports & configuration  <- EDIT FILE PATHS AND OUTPUT DIR HERE
1.  Load & describe data
2.  Per-track UMAP clustering (2D)
3.  Per-track Pareto-front analysis (minimise runtime_ms & veerScore)
4.  3-track overlay UMAP (shared embedding)
5.  Parameter evolution across generations
6.  Track-skew mathematical model
7.  Save all figures

Install dependencies (run once in your terminal):
    pip install umap-learn scikit-learn scipy matplotlib seaborn pandas numpy
"""

# ==============================================================================
# 0.  CONFIGURATION  -- edit these paths before running
# ==============================================================================
TRACK_PATHS = {
    1: r"C:\Users\xxx\Downloads\track1.csv",
    2: r"C:\Users\xxx\Downloads\track2.csv",
    3: r"C:\Users\xxx\Downloads\track3.csv",
}

# Folder where PNGs will be saved
OUTPUT_DIR = r"C:\Users\xxx\Downloads\micromouse_figures"

# Set to False if you don't want pop-up windows (e.g. running headless)
SHOW_PLOTS = True

# Number of UMAP clusters per track
N_CLUSTERS = 5

# ==============================================================================
# Imports
# ==============================================================================
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
import seaborn as sns
from scipy.stats import gaussian_kde, skew
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import umap

os.makedirs(OUTPUT_DIR, exist_ok=True)

# -- Global style: clean white background -------------------------------------
plt.rcParams.update({
    "figure.facecolor"  : "white",
    "axes.facecolor"    : "white",
    "text.color"        : "#111111",
    "axes.labelcolor"   : "#111111",
    "xtick.color"       : "#333333",
    "ytick.color"       : "#333333",
    "axes.edgecolor"    : "#cccccc",
    "grid.color"        : "#e5e5e5",
    "grid.linewidth"    : 0.7,
    "legend.framealpha" : 0.92,
    "legend.edgecolor"  : "#cccccc",
    "font.family"       : "DejaVu Sans",
    "axes.titleweight"  : "bold",
})
sns.set_style("whitegrid")

# -- Colour palettes ----------------------------------------------------------
PALETTE = {
    1: [(0.85, 0.15, 0.15),   # red
        (0.95, 0.50, 0.05),   # orange
        (0.80, 0.65, 0.00),   # gold
        (0.85, 0.20, 0.55),   # pink
        (0.60, 0.05, 0.30)],  # crimson
    2: [(0.10, 0.45, 0.85),   # blue
        (0.10, 0.65, 0.35),   # green
        (0.45, 0.10, 0.80),   # purple
        (0.05, 0.65, 0.75),   # teal
        (0.25, 0.35, 0.80)],  # periwinkle
    3: [(0.75, 0.50, 0.00),   # amber
        (0.00, 0.60, 0.40),   # emerald
        (0.80, 0.25, 0.00),   # burnt orange
        (0.35, 0.65, 0.10),   # olive
        (0.65, 0.10, 0.70)],  # violet
}
TRACK_ACCENT  = {1: "#d62728", 2: "#1f77b4", 3: "#e6a817"}
TRACK_ACCENT2 = {1: "#ff9999", 2: "#99bbff", 3: "#ffe099"}

FEATURE_COLS = [
    "kp", "kd", "base_speed", "min_base_speed",
    "corner1", "corner2", "corner3",
    "brake_pwr", "runtime_ms", "veerScore",
]

# ==============================================================================
# 1.  Load data
# ==============================================================================
def load(path):
    df = pd.read_csv(path).dropna().reset_index(drop=True)
    return df[df.finish == 1].reset_index(drop=True)

track = {t: load(p) for t, p in TRACK_PATHS.items()}

print("Loaded finished runs:")
for t, df in track.items():
    print(f"  Track {t}: {len(df)} rows | "
          f"runtime {df['runtime_ms'].min()/1000:.2f}s - {df['runtime_ms'].max()/1000:.2f}s | "
          f"veer {df['veerScore'].min():.2f} - {df['veerScore'].max():.2f}")

# ==============================================================================
# Helper functions
# ==============================================================================
def kde_density(X):
    """2-D KDE density scores for an (n, 2) array."""
    if len(X) < 4:
        return np.ones(len(X))
    return gaussian_kde(X.T)(X.T)


def brightness_colors(density, base_rgb, alpha_lo=0.15, alpha_hi=0.95):
    """Map density values to RGBA colours with varying alpha."""
    n = (density - density.min()) / (np.ptp(density) + 1e-9)
    alphas = alpha_lo + n * (alpha_hi - alpha_lo)
    return np.column_stack([np.tile(base_rgb, (len(alphas), 1)), alphas])


def pareto_front(costs):
    """Return boolean mask of non-dominated rows (minimisation, 2-objective)."""
    n = len(costs)
    dominated = np.zeros(n, dtype=bool)
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            if np.all(costs[j] <= costs[i]) and np.any(costs[j] < costs[i]):
                dominated[i] = True
                break
    return ~dominated


def cluster_label(cid, sub, name):
    return (f"{name} C{cid+1}\n"
            f"  kp={sub['kp'].mean():.3g}  kd={sub['kd'].mean():.3g}\n"
            f"  speed={int(sub['base_speed'].mean())}\n"
            f"  t={sub['runtime_ms'].mean()/1000:.2f}s  n={len(sub)}")


def fit_umap_shared(dfs):
    """Fit one UMAP on all tracks concatenated; return per-track slices."""
    arrays = [df[FEATURE_COLS].values for df in dfs]
    sizes  = [len(a) for a in arrays]
    scaled = StandardScaler().fit_transform(np.vstack(arrays))
    emb    = umap.UMAP(n_components=2, random_state=42,
                       min_dist=0.1, n_neighbors=20).fit_transform(scaled)
    splits, cursor = [], 0
    for s in sizes:
        splits.append(emb[cursor:cursor+s])
        cursor += s
    return splits


def save_fig(fig, name):
    path = os.path.join(OUTPUT_DIR, name)
    fig.savefig(path, dpi=150, bbox_inches="tight", facecolor="white")
    print(f"  Saved -> {path}")
    if SHOW_PLOTS:
        plt.show()
    plt.close(fig)


# ==============================================================================
# 2.  Per-track UMAP + KMeans clustering
# ==============================================================================
print("\nFitting per-track UMAP embeddings ...")
embeddings = {}
labels_km  = {}

for t, df in track.items():
    sc  = StandardScaler()
    emb = umap.UMAP(n_components=2, random_state=42, min_dist=0.1,
                    n_neighbors=min(20, len(df) - 1)
                    ).fit_transform(sc.fit_transform(df[FEATURE_COLS].values))
    km = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=15).fit(emb)
    embeddings[t] = emb
    labels_km[t]  = km.labels_
    print(f"  Track {t} done.")

# -- Figure A: per-track UMAP side by side ------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(21, 7), dpi=150, facecolor="white")
fig.suptitle("Per-Track UMAP Clustering  (finished runs only)",
             fontsize=18, fontweight="bold", color="#111111", y=1.01)

for t, ax in zip([1, 2, 3], axes):
    ax.set_facecolor("white")
    emb     = embeddings[t]
    lbs     = labels_km[t]
    df      = track[t]
    colors  = PALETTE[t]
    patches = []

    for cid in range(N_CLUSTERS):
        mask = lbs == cid
        pts  = emb[mask]
        if not mask.any():
            continue
        dens  = kde_density(pts)
        idx   = dens.argsort()
        pcols = brightness_colors(dens[idx], colors[cid])
        ax.scatter(pts[idx, 0], pts[idx, 1], c=pcols, s=16,
                   edgecolors="none", zorder=2)
        cx, cy = pts[:, 0].mean(), pts[:, 1].mean()
        ax.annotate(f"C{cid+1}", (cx, cy), fontsize=8.5, color="white",
                    ha="center", va="center", zorder=4,
                    bbox=dict(boxstyle="round,pad=0.28",
                              fc=colors[cid], alpha=0.92, ec="none"))
        patches.append(mpatches.Patch(color=colors[cid],
                                      label=cluster_label(cid, df[mask], f"T{t}")))

    ax.set_title(f"Track {t}", fontsize=14, color="#111111")
    ax.set_xlabel("UMAP-1", fontsize=10)
    ax.set_ylabel("UMAP-2", fontsize=10)
    ax.legend(handles=patches, fontsize=5.8, loc="upper right",
              facecolor="white", title=f"Track {t} clusters", title_fontsize=7,
              edgecolor="#cccccc")
    ax.grid(True, alpha=0.4)

plt.tight_layout()
save_fig(fig, "fig_A_per_track_umap.png")

# ==============================================================================
# 3.  Pareto-front analysis  (minimise runtime_ms AND veerScore)
# ==============================================================================
print("\nComputing Pareto fronts ...")
pareto_masks = {}
pareto_stats = {}

for t, df in track.items():
    costs = df[["runtime_ms", "veerScore"]].values
    pmask = pareto_front(costs)
    pareto_masks[t] = pmask
    psub = df[pmask]
    pareto_stats[t] = {
        "n"           : int(pmask.sum()),
        "mean_runtime": psub["runtime_ms"].mean(),
        "min_runtime" : psub["runtime_ms"].min(),
        "mean_veer"   : psub["veerScore"].mean(),
        "mean_kp"     : psub["kp"].mean(),
        "mean_kd"     : psub["kd"].mean(),
        "mean_speed"  : psub["base_speed"].mean(),
        "mean_minspd" : psub["min_base_speed"].mean(),
        "mean_corner1": psub["corner1"].mean(),
        "mean_corner2": psub["corner2"].mean(),
        "mean_corner3": psub["corner3"].mean(),
        "mean_brake"  : psub["brake_pwr"].mean(),
    }
    print(f"  Track {t}: {pmask.sum()} Pareto-optimal runs | "
          f"best time {psub['runtime_ms'].min()/1000:.2f}s | "
          f"min veer {psub['veerScore'].min():.2f}")

# -- Figure B: UMAP with Pareto highlighted -----------------------------------
fig, axes = plt.subplots(1, 3, figsize=(21, 7), dpi=150, facecolor="white")
fig.suptitle("UMAP Clusters + Pareto-Optimal Runs Highlighted",
             fontsize=18, fontweight="bold", y=1.01)

for t, ax in zip([1, 2, 3], axes):
    ax.set_facecolor("white")
    emb = embeddings[t]
    lbs = labels_km[t]
    pm  = pareto_masks[t]
    acc = TRACK_ACCENT[t]

    ax.scatter(emb[:, 0], emb[:, 1], c="#cccccc", s=9,
               edgecolors="none", zorder=1, alpha=0.6)

    for cid in range(N_CLUSTERS):
        mask = lbs == cid
        if not mask.any():
            continue
        cx, cy = emb[mask, 0].mean(), emb[mask, 1].mean()
        ax.annotate(f"C{cid+1}", (cx, cy), fontsize=7.5, color="white",
                    ha="center", va="center", zorder=3,
                    bbox=dict(boxstyle="round,pad=0.22",
                              fc=PALETTE[t][cid], alpha=0.88, ec="none"))

    ax.scatter(emb[pm, 0], emb[pm, 1], c=acc, s=55,
               edgecolors="white", linewidths=0.7, zorder=5,
               label=f"Pareto-optimal  (n={pm.sum()})")

    ax.set_title(f"Track {t}", fontsize=14)
    ax.set_xlabel("UMAP-1", fontsize=10)
    ax.set_ylabel("UMAP-2", fontsize=10)
    ax.legend(fontsize=9, facecolor="white", edgecolor="#cccccc")
    ax.grid(True, alpha=0.4)

plt.tight_layout()
save_fig(fig, "fig_B_umap_pareto.png")

# -- Figure C: Pareto fronts in objective space --------------------------------
fig, ax = plt.subplots(figsize=(11, 7), dpi=150, facecolor="white")
ax.set_facecolor("white")

for t, df in track.items():
    pm  = pareto_masks[t]
    acc = TRACK_ACCENT[t]
    lt  = TRACK_ACCENT2[t]

    ax.scatter(df["runtime_ms"] / 1000, df["veerScore"],
               c=lt, s=12, alpha=0.35, edgecolors="none",
               label=f"Track {t} - all runs")

    psub = df[pm].sort_values("runtime_ms")
    ax.scatter(psub["runtime_ms"] / 1000, psub["veerScore"],
               c=acc, s=70, edgecolors="white", linewidths=0.8, zorder=5)
    ax.plot(psub["runtime_ms"] / 1000, psub["veerScore"],
            c=acc, lw=2.0, zorder=4, label=f"Track {t} - Pareto front")

ax.set_xlabel("Runtime (s)", fontsize=13)
ax.set_ylabel("Veer Score", fontsize=13)
ax.set_title("Pareto Fronts in Objective Space - All Tracks",
             fontsize=15, fontweight="bold")
ax.legend(fontsize=9, facecolor="white", edgecolor="#cccccc")
ax.grid(True, alpha=0.4)
plt.tight_layout()
save_fig(fig, "fig_C_pareto_objective.png")

# ==============================================================================
# 4.  3-track overlay UMAP  (shared embedding space)
# ==============================================================================
print("\nFitting 3-track shared UMAP embedding ...")
splits     = fit_umap_shared([track[1], track[2], track[3]])
emb_shared = {t + 1: splits[t] for t in range(3)}

fig, ax = plt.subplots(figsize=(14, 9), dpi=150, facecolor="white")
ax.set_facecolor("white")
legend_handles = []

for t in [1, 2, 3]:
    emb    = emb_shared[t]
    lbs    = KMeans(n_clusters=N_CLUSTERS, random_state=42,
                    n_init=15).fit_predict(emb)
    pm     = pareto_masks[t]
    colors = PALETTE[t]
    acc    = TRACK_ACCENT[t]

    for cid in range(N_CLUSTERS):
        mask = lbs == cid
        if not mask.any():
            continue
        pts   = emb[mask]
        dens  = kde_density(pts)
        idx   = dens.argsort()
        pcols = brightness_colors(dens[idx], colors[cid], alpha_lo=0.12)
        ax.scatter(pts[idx, 0], pts[idx, 1], c=pcols, s=10,
                   edgecolors="none", zorder=1)

    ax.scatter(emb[pm, 0], emb[pm, 1], c=acc, s=60,
               edgecolors="white", linewidths=0.7, zorder=6)

    legend_handles += [
        Line2D([0], [0], marker="o", color="w", markerfacecolor=acc,
               markersize=9, label=f"Track {t} - Pareto-optimal"),
        mpatches.Patch(facecolor=colors[0], alpha=0.75,
                       label=f"Track {t} - clusters"),
    ]

ax.set_title("3-Track Overlay UMAP  (shared embedding)  dot = Pareto-optimal",
             fontsize=14, fontweight="bold")
ax.set_xlabel("UMAP-1", fontsize=11)
ax.set_ylabel("UMAP-2", fontsize=11)
ax.legend(handles=legend_handles, fontsize=8.5, facecolor="white",
          edgecolor="#cccccc", loc="upper right")
ax.grid(True, alpha=0.35)
plt.tight_layout()
save_fig(fig, "fig_D_3track_overlay.png")

# ==============================================================================
# 5.  Parameter evolution across runs  (generation proxy)
# ==============================================================================
print("\nPlotting parameter evolution ...")

PARAMS_EVO = [
    ("kp",            "kp  (prop. gain)"),
    ("kd",            "kd  (deriv. gain)"),
    ("base_speed",    "Base Speed"),
    ("min_base_speed","Min Speed"),
    ("corner1",       "Corner 1 stutter"),
    ("corner2",       "Corner 2 stutter"),
    ("corner3",       "Corner 3 stutter"),
    ("brake_pwr",     "Brake Power"),
    ("runtime_ms",    "Runtime (ms)"),
    ("veerScore",     "Veer Score"),
]

fig, axes = plt.subplots(2, 5, figsize=(26, 10), dpi=130, facecolor="white")
fig.suptitle("Parameter Evolution Across Runs  (rolling mean + Pareto points)",
             fontsize=16, fontweight="bold", y=1.01)

for pi, (param, plabel) in enumerate(PARAMS_EVO):
    ax = axes.flatten()[pi]
    ax.set_facecolor("white")

    for t in [1, 2, 3]:
        df  = track[t]
        y   = df[param].values
        x   = np.arange(len(y))
        acc = TRACK_ACCENT[t]
        lt  = TRACK_ACCENT2[t]

        ax.scatter(x, y, c=lt, s=4, alpha=0.3, edgecolors="none")
        roll = pd.Series(y).rolling(15, center=True, min_periods=5).mean()
        ax.plot(x, roll, c=acc, lw=1.8, label=f"T{t}")
        pm = pareto_masks[t]
        ax.scatter(np.where(pm)[0], y[pm], c=acc, s=25,
                   edgecolors="#333", linewidths=0.35, zorder=5)

    ax.set_title(plabel, fontsize=10)
    ax.set_xlabel("Run index", fontsize=7.5)
    ax.grid(True, alpha=0.35)
    if pi == 0:
        ax.legend(fontsize=7.5, facecolor="white", edgecolor="#cccccc")

plt.tight_layout()
save_fig(fig, "fig_E_param_evolution.png")

# ==============================================================================
# 6.  Track-skew mathematical model
# ==============================================================================
print("\nBuilding skew-parameter model ...")

# -- 6a. Skew metric: Fisher skewness of runtime + veer distributions ---------
skew_metric = {}
for t, df in track.items():
    rt_skew  = float(skew(df["runtime_ms"]))
    vr_skew  = float(skew(df["veerScore"]))
    combined = (rt_skew + vr_skew) / 2.0
    skew_metric[t] = combined
    print(f"  Track {t}: runtime_skew={rt_skew:.3f}  veer_skew={vr_skew:.3f}  "
          f"combined={combined:.3f}")

# -- 6b. Pareto-optimal mean values per parameter per track -------------------
TARGET_PARAMS = [
    "kp", "kd", "base_speed", "min_base_speed",
    "corner1", "corner2", "corner3", "brake_pwr",
]
stat_key = {
    "kp"            : "mean_kp",
    "kd"            : "mean_kd",
    "base_speed"    : "mean_speed",
    "min_base_speed": "mean_minspd",
    "corner1"       : "mean_corner1",
    "corner2"       : "mean_corner2",
    "corner3"       : "mean_corner3",
    "brake_pwr"     : "mean_brake",
}
pareto_means = {
    p: np.array([pareto_stats[t][stat_key[p]] for t in [1, 2, 3]])
    for p in TARGET_PARAMS
}

param_labels_full = {
    "kp"            : "kp  (prop. gain)",
    "kd"            : "kd  (deriv. gain)",
    "base_speed"    : "Base Speed",
    "min_base_speed": "Min Speed",
    "corner1"       : "Corner 1 stutter",
    "corner2"       : "Corner 2 stutter",
    "corner3"       : "Corner 3 stutter",
    "brake_pwr"     : "Brake Power",
}

# -- 6c. Fit linear & quadratic vs. track number ------------------------------
track_ids = np.array([1.0, 2.0, 3.0])

def fit_and_report(x, y):
    results = {}
    for deg, name in [(1, "linear"), (2, "quadratic")]:
        coeffs = np.polyfit(x, y, deg)
        y_pred = np.polyval(coeffs, x)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - y.mean()) ** 2)
        r2 = 1.0 - ss_res / ss_tot if ss_tot > 1e-10 else 1.0
        results[name] = {"coeffs": coeffs, "r2": r2}
    best = max(results, key=lambda k: results[k]["r2"])
    return results, best

fit_results = {}
for p in TARGET_PARAMS:
    fr, best = fit_and_report(track_ids, pareto_means[p])
    fit_results[p] = {"fits": fr, "best": best}

# Print equation table
print("\n-- Fitted equations  (optimal param = f(track number T)) --")
print(f"{'Parameter':<20} {'Model':<10} {'Equation':<50} R2")
print("-" * 90)
for p in TARGET_PARAMS:
    fr   = fit_results[p]["fits"]
    best = fit_results[p]["best"]
    c    = fr[best]["coeffs"]
    r2   = fr[best]["r2"]
    if len(c) == 2:
        eq = f"{p} = {c[0]:+.4f}*T  {c[1]:+.4f}"
    else:
        eq = f"{p} = {c[0]:+.4f}*T^2  {c[1]:+.4f}*T  {c[2]:+.4f}"
    print(f"{p:<20} {best:<10} {eq:<50} {r2:.4f}")

# -- Figure F: regression model plots -----------------------------------------
x_dense = np.linspace(0.8, 3.2, 300)

fig = plt.figure(figsize=(22, 12), dpi=130, facecolor="white")
fig.suptitle("Optimal Parameter vs. Track Number - Regression Models",
             fontsize=17, fontweight="bold", y=1.005)
gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.50, wspace=0.38)

for i, p in enumerate(TARGET_PARAMS):
    row, col = divmod(i, 4)
    ax = fig.add_subplot(gs[row, col])
    ax.set_facecolor("white")

    y    = pareto_means[p]
    fr   = fit_results[p]["fits"]
    best = fit_results[p]["best"]

    for t in [1, 2, 3]:
        ax.scatter(t, y[t - 1], c=TRACK_ACCENT[t], s=110,
                   zorder=5, edgecolors="#333", linewidths=0.7)
        ax.text(t + 0.07, y[t - 1], f"T{t}", fontsize=7.5,
                color=TRACK_ACCENT[t], va="center")

    for name, (ls, lc, lw) in [
        ("linear",    ("--", "#999999", 1.5)),
        ("quadratic", ("-",  "#1f77b4", 2.0)),
    ]:
        c  = fr[name]["coeffs"]
        r2 = fr[name]["r2"]
        ax.plot(x_dense, np.polyval(c, x_dense), ls=ls, c=lc, lw=lw,
                label=f"{name}  R2={r2:.3f}")

    c = fr[best]["coeffs"]
    if len(c) == 2:
        eq_str = f"{c[0]:+.3f}*T\n{c[1]:+.3f}"
    else:
        eq_str = f"{c[0]:+.3f}*T^2\n{c[1]:+.3f}*T  {c[2]:+.3f}"

    ax.text(0.04, 0.04, eq_str, transform=ax.transAxes,
            fontsize=6.5, color="#333333", va="bottom",
            bbox=dict(fc="white", alpha=0.85, ec="#cccccc", boxstyle="round,pad=0.3"))

    ax.set_title(param_labels_full[p], fontsize=10)
    ax.set_xlabel("Track number T", fontsize=8)
    ax.set_xticks([1, 2, 3])
    ax.legend(fontsize=6.5, facecolor="white", edgecolor="#cccccc", loc="best")
    ax.grid(True, alpha=0.4)

save_fig(fig, "fig_F_param_model.png")

# -- Figure G: skew distributions & skew vs. track number --------------------
fig, axes = plt.subplots(1, 2, figsize=(15, 6), dpi=150, facecolor="white")

ax = axes[0]
ax.set_facecolor("white")
for t in [1, 2, 3]:
    rt = track[t]["runtime_ms"].values / 1000
    sns.kdeplot(rt, ax=ax, color=TRACK_ACCENT[t], fill=True, alpha=0.20,
                label=f"Track {t}  (skew = {skew(rt):.2f})", linewidth=2.0)
ax.set_xlabel("Runtime (s)", fontsize=12)
ax.set_ylabel("Density", fontsize=12)
ax.set_title("Runtime Distributions per Track\n(skewness = curve-difficulty proxy)",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=9, facecolor="white", edgecolor="#cccccc")
ax.grid(True, alpha=0.4)

ax = axes[1]
ax.set_facecolor("white")
sv = [skew_metric[t] for t in [1, 2, 3]]
for t in [1, 2, 3]:
    ax.scatter(t, sv[t - 1], c=TRACK_ACCENT[t], s=160, zorder=5,
               edgecolors="#333", linewidths=0.7, label=f"Track {t}")
coeffs_skew = np.polyfit([1, 2, 3], sv, 1)
y_fit  = np.polyval(coeffs_skew, x_dense)
ss_res = np.sum((np.array(sv) - np.polyval(coeffs_skew, [1, 2, 3])) ** 2)
ss_tot = np.sum((np.array(sv) - np.mean(sv)) ** 2)
r2_sk  = 1 - ss_res / max(ss_tot, 1e-10)
ax.plot(x_dense, y_fit, "--", c="#555555", lw=1.8,
        label=f"Linear fit  R2={r2_sk:.3f}")
ax.set_xlabel("Track number", fontsize=12)
ax.set_ylabel("Combined skew metric", fontsize=12)
ax.set_title("Track Skew Metric vs. Track Number", fontsize=12, fontweight="bold")
ax.set_xticks([1, 2, 3])
ax.legend(fontsize=9, facecolor="white", edgecolor="#cccccc")
ax.grid(True, alpha=0.4)

plt.tight_layout()
save_fig(fig, "fig_G_skew_model.png")

# -- Figure H: normalised Pareto-mean heatmap ---------------------------------
fig, ax = plt.subplots(figsize=(12, 5), dpi=150, facecolor="white")
ax.set_facecolor("white")

norm_params = {
    p: (pareto_means[p] - pareto_means[p].min()) /
       (np.ptp(pareto_means[p]) + 1e-9)
    for p in TARGET_PARAMS
}
heat_data = np.array([norm_params[p] for p in TARGET_PARAMS])

im = ax.imshow(heat_data, aspect="auto", cmap="YlOrRd", vmin=0, vmax=1)
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(["Track 1", "Track 2", "Track 3"], fontsize=11)
ax.set_yticks(range(len(TARGET_PARAMS)))
ax.set_yticklabels([param_labels_full[p] for p in TARGET_PARAMS], fontsize=9)
ax.set_title("Normalised Pareto-Optimal Mean Parameters per Track",
             fontsize=13, fontweight="bold")

cbar = plt.colorbar(im, ax=ax)
cbar.set_label("Normalised value  (0 = min, 1 = max)", fontsize=9)

for yi, p in enumerate(TARGET_PARAMS):
    for xi, t in enumerate([1, 2, 3]):
        raw = pareto_means[p][xi]
        fmt = f"{raw:.3f}" if raw < 1 else f"{int(raw)}"
        text_color = "white" if heat_data[yi, xi] > 0.6 else "#111111"
        ax.text(xi, yi, fmt, ha="center", va="center",
                fontsize=8.5, color=text_color, fontweight="bold")

plt.tight_layout()
save_fig(fig, "fig_H_pareto_heatmap.png")

# ==============================================================================
# Final equation summary
# ==============================================================================
print("\n" + "=" * 72)
print("  MATHEMATICAL MODEL SUMMARY")
print("  Optimal parameter as a function of track number  T in {1, 2, 3}")
print("=" * 72)
for p in TARGET_PARAMS:
    fr   = fit_results[p]["fits"]
    best = fit_results[p]["best"]
    c    = fr[best]["coeffs"]
    r2   = fr[best]["r2"]
    if len(c) == 2:
        eq = f"{param_labels_full[p]:<22}  ~  {c[0]:+.4f}*T  {c[1]:+.4f}"
    else:
        eq = (f"{param_labels_full[p]:<22}  ~  {c[0]:+.4f}*T^2  "
              f"{c[1]:+.4f}*T  {c[2]:+.4f}")
    print(f"  {eq:<60}  (R2={r2:.4f}, {best})")
print("=" * 72)
print(f"\nAll figures saved to:  {OUTPUT_DIR}")


Loaded finished runs:
  Track 1: 323 rows | runtime 11.22s - 24.80s | veer 10.10 - 38.81
  Track 2: 341 rows | runtime 11.41s - 23.55s | veer 12.15 - 29.50
  Track 3: 301 rows | runtime 8.60s - 24.95s | veer 9.69 - 37.27

Fitting per-track UMAP embeddings ...
  Track 1 done.
  Track 2 done.
  Track 3 done.
  Saved -> C:\Users\sunla\Downloads\micromouse_figures\fig_A_per_track_umap.png

Computing Pareto fronts ...
  Track 1: 18 Pareto-optimal runs | best time 11.22s | min veer 10.10
  Track 2: 24 Pareto-optimal runs | best time 11.41s | min veer 12.15
  Track 3: 8 Pareto-optimal runs | best time 8.60s | min veer 9.69
  Saved -> C:\Users\sunla\Downloads\micromouse_figures\fig_B_umap_pareto.png
  Saved -> C:\Users\sunla\Downloads\micromouse_figures\fig_C_pareto_objective.png

Fitting 3-track shared UMAP embedding ...
  Saved -> C:\Users\sunla\Downloads\micromouse_figures\fig_D_3track_overlay.png

Plotting parameter evolution ...
  Saved -> C:\Users\sunla\Downloads\micromouse_figures\fig_E